In [1]:
# import June's 'clean.sales.detail'
import pandas as pd

path = '/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/VK_query_results/Jun_2018/clean.sales.detail.csv'
june_sale = pd.read_csv(path)

In [2]:
june_sale.head()

,sitenumber,alohachecknumber,businessdate,timesold,alohaitemposcode,alohaitemname,ordermode,totalprice,totalqty,quickcomboid,quickcomboupsell,w.day,weeknum,fkstoreid,checknumber,uid
0,1003461,30071,2018-06-30,19,400113,LARGE SPRITE,GoJek,9857.0,1,117,1,Sat,5,23616,30071,23616.2018-06-30.30071
1,1003461,30019,2018-06-11,15,213,CHEESE BURGER,GoJek,20909.0,1,0,0,Mon,2,23616,30019,23616.2018-06-11.30019
2,1003461,30083,2018-06-11,21,400028,FLOAT MILO,Take Away,10000.0,1,447,0,Mon,2,23616,30083,23616.2018-06-11.30083
3,1003461,40048,2018-06-23,18,7594,TakeAway Charge,Take Away,909.0,1,0,0,Sat,4,23616,40048,23616.2018-06-23.40048
4,1003461,10060,2018-06-14,18,711,MEDIUM FRIES,GoJek,11406.0,1,111,0,Thu,2,23616,10060,23616.2018-06-14.10060


In [3]:
print(june_sale.info())
print(june_sale.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7461020 entries, 0 to 7461019
Data columns (total 16 columns):
 #   Column            Dtype  
---  ------            -----  
 0   sitenumber        int64  
 1   alohachecknumber  int64  
 2   businessdate      object 
 3   timesold          int64  
 4   alohaitemposcode  int64  
 5   alohaitemname     object 
 6   ordermode         object 
 7   totalprice        float64
 8   totalqty          int64  
 9   quickcomboid      int64  
 10  quickcomboupsell  int64  
 11  w.day             object 
 12  weeknum           int64  
 13  fkstoreid         int64  
 14  checknumber       int64  
 15  uid               object 
dtypes: float64(1), int64(10), object(5)
memory usage: 910.8+ MB
None
(7461020, 16)


In [4]:
print(june_sale.describe())

         sitenumber  alohachecknumber      timesold  alohaitemposcode  \
count  7.461020e+06      7.461020e+06  7.461020e+06      7.461020e+06   
mean   1.005901e+06      3.007825e+04  1.591948e+01      1.884139e+05   
std    1.590576e+03      2.159982e+04  3.853699e+00      2.993567e+05   
min    1.003461e+06      1.000100e+04  0.000000e+00      1.100000e+02   
25%    1.004671e+06      2.002100e+04  1.300000e+01      3.220000e+02   
50%    1.005568e+06      3.001500e+04  1.600000e+01      7.130000e+02   
75%    1.006335e+06      4.001700e+04  1.900000e+01      4.000360e+05   
max    1.009669e+06      1.205240e+05  2.300000e+01      9.012600e+05   

         totalprice      totalqty  quickcomboid  quickcomboupsell  \
count  7.461020e+06  7.461020e+06  7.461020e+06      7.461020e+06   
mean   1.283133e+04  1.412969e+00  2.480718e+02      2.934907e-02   
std    1.885825e+04  1.585052e+00  2.411407e+02      1.687830e-01   
min   -8.490500e+05  1.000000e+00  0.000000e+00      0.000000e+00 

In [5]:
# Filter the dataset for the platform order modes
platform_order = june_sale[june_sale['ordermode'].isin(['GoJek', 'Grab'])]

# Calculate the percentage of transactions with upselling
upselling_percentage = (platform_order['quickcomboupsell'].sum() / len(platform_order)) * 100

print(f"Percentage of transactions with upselling through platform: {upselling_percentage:.2f}%")


# Calculate the total revenue from upselling transactions
upsell_revenue = june_sale[june_sale['quickcomboupsell'] == 1]['totalprice'].sum()

# Calculate the total revenue from all transactions
total_revenue = june_sale['totalprice'].sum()

# Calculate the proportion of revenue from upselling
proportion_upsell_revenue = upsell_revenue / total_revenue

print(f"Proportion of revenue from upselling: {proportion_upsell_revenue:.2%}")



Percentage of transactions with upselling through platform: 0.85%
Proportion of revenue from upselling: 2.06%


In [6]:
# Filter the dataset for the non-platform order modes
non_platform_order = june_sale[~june_sale['ordermode'].isin(['GoJek', 'Grab'])]

# Calculate the percentage of transactions with upselling
upselling_percentage_non_platform = (non_platform_order['quickcomboupsell'].sum() / len(non_platform_order)) * 100

print(f"Percentage of transactions with upselling through non-platform channels: {upselling_percentage_non_platform:.2f}%")

Percentage of transactions with upselling through non-platform channels: 3.12%


In [7]:
##################BEGIN############################

In [8]:
def revenueProp(fileyear):
    filepath = f"/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/VK_query_results/{fileyear}/clean.sales.detail.csv"
    file2 = pd.read_csv(filepath)
    
    # Filtering out the TakeAway Charge
    file2 = file2[file2['alohaitemname'] != "TakeAway Charge"]
    
    # Calculating Total Price for upsell
    summedPrices = file2.groupby(['businessdate', 'fkstoreid', 'quickcomboupsell']).agg({'totalprice': 'sum'}).reset_index()
    summedPrices.rename(columns={'totalprice': 'upsell'}, inplace=True)
    
    # Summing upselling item revenue
    totalPrices = summedPrices.groupby(['businessdate', 'fkstoreid']).agg({'upsell': 'sum'}).reset_index()
    totalPrices.rename(columns={'upsell': 'totalprices'}, inplace=True)
    
    # Calculating Proportion
    newdf = pd.merge(totalPrices, summedPrices, on=['businessdate', 'fkstoreid'])
    newdf['proportionUpsellrevenue'] = newdf['upsell'] / newdf['totalprices']
    
    # Housekeeping the DataFrame
    newdf = newdf[newdf['quickcomboupsell'] == 1]
    newdf = newdf[['businessdate', 'fkstoreid', 'proportionUpsellrevenue']]
    
    # Tabulating statistics for Proportion of upsell Revenue
    print(fileyear)
    print(f"Mean: {newdf['proportionUpsellrevenue'].mean()}")
    print(f"Median: {newdf['proportionUpsellrevenue'].median()}")
    print(f"IQR: {newdf['proportionUpsellrevenue'].quantile(0.75) - newdf['proportionUpsellrevenue'].quantile(0.25)}")
    
    return newdf


In [9]:
fileyear = "Jan_2018"
pUpsellrevenue = revenueProp(fileyear)

lst = ["Feb_2018", "Mar_2018", "Apr_2018", "May_2018", "Jun_2018", "Jul_2018", "Aug_2018", "Sep_2018", "Oct_2018", "Nov_2018", "Dec_2018"]
for i in lst:
    dfToMerge = revenueProp(i)
    pUpsellrevenue = pd.concat([pUpsellrevenue, dfToMerge])


Jan_2018
Mean: 0.02751106499127068
Median: 0.02263571234723508
IQR: 0.01978518212283739
Feb_2018
Mean: 0.024780618381374048
Median: 0.019939597939507194
IQR: 0.01744382407476196
Mar_2018
Mean: 0.021784147943032815
Median: 0.018123729809238744
IQR: 0.014891244230988485
Apr_2018
Mean: 0.02317329483720363
Median: 0.019998086932075443
IQR: 0.016131616543326333
May_2018
Mean: 0.02246943783192419
Median: 0.0178634250681095
IQR: 0.01628546289666883
Jun_2018
Mean: 0.02040008447522971
Median: 0.015862549622882333
IQR: 0.014618280670289675
Jul_2018
Mean: 0.028917012063778644
Median: 0.0205288308935813
IQR: 0.01916304872067711
Aug_2018
Mean: 0.033711497012135036
Median: 0.02334884427404335
IQR: 0.02225464113175157
Sep_2018
Mean: 0.04162891563793094
Median: 0.03268437700266742
IQR: 0.02638316513196629
Oct_2018
Mean: 0.04380952021213265
Median: 0.03521434745613038
IQR: 0.022138774691646552
Nov_2018
Mean: 0.05603865621976071
Median: 0.04619697990255374
IQR: 0.03291705326048614
Dec_2018
Mean: 0.06202

In [10]:
print(f"Mean: {pUpsellrevenue['proportionUpsellrevenue'].mean()}")
print(f"Median: {pUpsellrevenue['proportionUpsellrevenue'].median()}")
print(f"IQR: {pUpsellrevenue['proportionUpsellrevenue'].quantile(0.75) - pUpsellrevenue['proportionUpsellrevenue'].quantile(0.25)}")
print(f"SD: {pUpsellrevenue['proportionUpsellrevenue'].std()}")
print(pUpsellrevenue['proportionUpsellrevenue'].quantile([0.0, 0.25, 0.5, 0.75, 1.0]))

Mean: 0.03510522602768723
Median: 0.026206130255482837
IQR: 0.02681869050200134
SD: 0.030859388504289376
0.00    0.000147
0.25    0.015754
0.50    0.026206
0.75    0.042572
1.00    0.254986
Name: proportionUpsellrevenue, dtype: float64


In [11]:
file = pUpsellrevenue

# Data Cleaning
file['businessdate'] = pd.to_datetime(file['businessdate'], format='%Y-%m-%d')

# Adding businessmonth, w.day, weeknum as factors
file['businessmonth'] = file['businessdate'].dt.month
file['w.day'] = file['businessdate'].dt.dayofweek
file['weeknum'] = ((file['businessdate'].dt.day - 1) // 7 + 1)

# Convert to categorical
file['businessmonth'] = file['businessmonth'].astype('category')
file['w.day'] = file['w.day'].astype('category')
file['weeknum'] = file['weeknum'].astype('category')

# Adding trend.var
file['trend.var'] = (file['businessdate'] - file['businessdate'].min()).dt.days + 1

In [12]:
print(file.head())
print(file.shape)
# 'proportionUpsellrevenue' is our dependent variable y 

  businessdate  fkstoreid  proportionUpsellrevenue businessmonth w.day  \
1   2018-01-01    16054.0                 0.038784             1     0   
3   2018-01-01    16227.0                 0.019313             1     0   
5   2018-01-01    16309.0                 0.018664             1     0   
7   2018-01-01    16702.0                 0.009755             1     0   
9   2018-01-01    17523.0                 0.023666             1     0   

  weeknum  trend.var  
1       1          1  
3       1          1  
5       1          1  
7       1          1  
9       1          1  
(35538, 7)


In [13]:
from datetime import datetime
import os

# Getting the public holidays data

path1 = '/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/rstudio-export-20230725/2018.public.holidays.indonesia.csv'
ph_list = pd.read_csv(path1)

# Housekeeping
ph_list['public_holiday'] = 1
ph_list['businessdate'] = pd.to_datetime(ph_list['date'], format='%m/%d/%y')
ph_list['ph_wday'] = ph_list['businessdate'].dt.day_name()
ph_list['ph_weekend'] = ph_list['public_holiday'] * ph_list['ph_wday'].isin(['Saturday', 'Sunday'])

# Selecting columns
tableToMerge = ph_list[['businessdate', 'public_holiday', 'ph_weekend']]

# Merging in PH Details
file_merged = pd.merge(file, tableToMerge, on='businessdate', how='left')

# Replacing NaN values with 0 in 'public_holiday' and 'ph_weekend'
file_merged['public_holiday'].fillna(0, inplace=True)
file_merged['ph_weekend'].fillna(0, inplace=True)

# Displaying the first few rows of the merged DataFrame
print(file_merged.head())


  businessdate  fkstoreid  proportionUpsellrevenue businessmonth w.day  \
0   2018-01-01    16054.0                 0.038784             1     0   
1   2018-01-01    16227.0                 0.019313             1     0   
2   2018-01-01    16309.0                 0.018664             1     0   
3   2018-01-01    16702.0                 0.009755             1     0   
4   2018-01-01    17523.0                 0.023666             1     0   

  weeknum  trend.var  public_holiday  ph_weekend  
0       1          1             1.0         0.0  
1       1          1             1.0         0.0  
2       1          1             1.0         0.0  
3       1          1             1.0         0.0  
4       1          1             1.0         0.0  


In [14]:
# Adding a dummy for the Ramadan period in 2018
file_merged['ramadan'] = 0

# Creating a mask for the Ramadan period
ramadan_start = '2018-05-16'
ramadan_end = '2018-06-14'
ramadan_mask = (file_merged['businessdate'] >= ramadan_start) & (file_merged['businessdate'] <= ramadan_end)

# Applying the mask to the DataFrame
file_merged.loc[ramadan_mask, 'ramadan'] = 1

# Optional: Creating a test DataFrame for Ramadan period
ramadan_test = file_merged[['businessdate', 'ramadan']].drop_duplicates()

# Displaying the first few rows of the merged DataFrame
print(file_merged.head())

  businessdate  fkstoreid  proportionUpsellrevenue businessmonth w.day  \
0   2018-01-01    16054.0                 0.038784             1     0   
1   2018-01-01    16227.0                 0.019313             1     0   
2   2018-01-01    16309.0                 0.018664             1     0   
3   2018-01-01    16702.0                 0.009755             1     0   
4   2018-01-01    17523.0                 0.023666             1     0   

  weeknum  trend.var  public_holiday  ph_weekend  ramadan  
0       1          1             1.0         0.0        0  
1       1          1             1.0         0.0        0  
2       1          1             1.0         0.0        0  
3       1          1             1.0         0.0        0  
4       1          1             1.0         0.0        0  


In [15]:
import pandas as pd
import os

# Getting the promotions data
path2 = '/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/new.GoJek.Promos.csv'
banners = pd.read_csv(path2)

# Converting 'businessdate' to datetime
banners['businessdate'] = pd.to_datetime(banners['businessdate'], format='%Y-%m-%d')

# Selecting specific columns from banners
bannersToMerge = banners[['fkstoreid', 'businessdate', 'banner.binary', 'go.banner.slide', 'banner.day']]

# Displaying the first few rows of bannersToMerge
print(bannersToMerge.head())


# Converting 'businessdate' in file_merged to datetime (if not already in this format)
file_merged['businessdate'] = pd.to_datetime(file_merged['businessdate'], format='%Y-%m-%d')

# Merging file_merged with bannersToMerge
file_merged = pd.merge(file_merged, bannersToMerge, on=['fkstoreid', 'businessdate'], how='left')

# Displaying the first few rows of the merged DataFrame
print(file_merged.head())

   fkstoreid businessdate  banner.binary  go.banner.slide  banner.day
0      16054   2018-01-01              0                0           0
1      16054   2018-01-02              0                0           0
2      16054   2018-01-03              0                0           0
3      16054   2018-01-04              0                0           0
4      16054   2018-01-05              0                0           0
  businessdate  fkstoreid  proportionUpsellrevenue businessmonth w.day  \
0   2018-01-01    16054.0                 0.038784             1     0   
1   2018-01-01    16227.0                 0.019313             1     0   
2   2018-01-01    16309.0                 0.018664             1     0   
3   2018-01-01    16702.0                 0.009755             1     0   
4   2018-01-01    17523.0                 0.023666             1     0   

  weeknum  trend.var  public_holiday  ph_weekend  ramadan  banner.binary  \
0       1          1             1.0         0.0        0  

In [16]:
import pandas as pd
import os

# Getting the dependence data
path3 = '/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/rstudio-export-20230725/2018.site.daily.platform.dependence.csv'
site_daily_dependence = pd.read_csv(path3)

# Housekeeping the Dependence Data
dependenceData = site_daily_dependence[['fkstoreid', 'businessdate', 'fourteen.day.disc.price.sales.dep']]

# Converting 'businessdate' to datetime
dependenceData['businessdate'] = pd.to_datetime(dependenceData['businessdate'])

# Displaying the first few rows of dependenceData
print(dependenceData.head())

   fkstoreid businessdate  fourteen.day.disc.price.sales.dep
0      16054   2018-01-01                                NaN
1      16054   2018-01-02                                NaN
2      16054   2018-01-03                                NaN
3      16054   2018-01-04                                NaN
4      16054   2018-01-05                                NaN


/var/folders/y2/30zds30n5gv0yf5nk5xk2c9c0000gn/T/ipykernel_52996/3414517140.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dependenceData['businessdate'] = pd.to_datetime(dependenceData['businessdate'])


In [17]:
# Merging the 2 dataframes
file_merged = pd.merge(file_merged, dependenceData, on=['fkstoreid', 'businessdate'])

# Displaying the first few rows of the merged DataFrame
print(file_merged.head())

  businessdate  fkstoreid  proportionUpsellrevenue businessmonth w.day  \
0   2018-01-01    16054.0                 0.038784             1     0   
1   2018-01-01    16227.0                 0.019313             1     0   
2   2018-01-01    16309.0                 0.018664             1     0   
3   2018-01-01    16702.0                 0.009755             1     0   
4   2018-01-01    17523.0                 0.023666             1     0   

  weeknum  trend.var  public_holiday  ph_weekend  ramadan  banner.binary  \
0       1          1             1.0         0.0        0            0.0   
1       1          1             1.0         0.0        0            0.0   
2       1          1             1.0         0.0        0            0.0   
3       1          1             1.0         0.0        0            0.0   
4       1          1             1.0         0.0        0            0.0   

   go.banner.slide  banner.day  fourteen.day.disc.price.sales.dep  
0              0.0         0.0

In [18]:
# Converting to categorical data
file_merged['fkstoreid'] = file_merged['fkstoreid'].astype('category')
#file_merged['sitenumber'] = file_merged['sitenumber'].astype('category')  # Assuming 'sitenumber' is in file_merged
file_merged['businessmonth'] = file_merged['businessmonth'].astype('category')  # Assuming 'businessmonth' is in file_merged
file_merged['w.day'] = file_merged['w.day'].astype('category')  # Assuming 'w.day' is in file_merged
file_merged['weeknum'] = file_merged['weeknum'].astype('category')

summary = file_merged.describe(include='all')

# Print the summary
print(summary)

# Number of Rows in the DataFrame
num_rows = len(file_merged)

# Print the number of rows
print(f"Number of Rows: {num_rows}")

               businessdate  fkstoreid  proportionUpsellrevenue  \
count                 34268    34268.0             34268.000000   
unique                  365       99.0                      NaN   
top     2018-07-02 00:00:00    23329.0                      NaN   
freq                     99      365.0                      NaN   
first   2018-01-01 00:00:00        NaN                      NaN   
last    2018-12-31 00:00:00        NaN                      NaN   
mean                    NaN        NaN                 0.034687   
std                     NaN        NaN                 0.030889   
min                     NaN        NaN                 0.000276   
25%                     NaN        NaN                 0.015457   
50%                     NaN        NaN                 0.025705   
75%                     NaN        NaN                 0.041842   
max                     NaN        NaN                 0.254986   

        businessmonth    w.day  weeknum     trend.var  public

/var/folders/y2/30zds30n5gv0yf5nk5xk2c9c0000gn/T/ipykernel_52996/3438004664.py:8: FutureWarning: Treating datetime data as categorical rather than numeric in `.describe` is deprecated and will be removed in a future version of pandas. Specify `datetime_is_numeric=True` to silence this warning and adopt the future behavior now.
  summary = file_merged.describe(include='all')


In [19]:
print(file_merged.columns)

#export it as excel
!pip install openpyxl
file_merged.to_excel('output_filename.xlsx', index=False, engine='openpyxl')

Index(['businessdate', 'fkstoreid', 'proportionUpsellrevenue', 'businessmonth',
       'w.day', 'weeknum', 'trend.var', 'public_holiday', 'ph_weekend',
       'ramadan', 'banner.binary', 'go.banner.slide', 'banner.day',
       'fourteen.day.disc.price.sales.dep'],
      dtype='object')


In [20]:
import numpy as np
# Regression for Proportion of Daily Revenue Generated from upsell
# Selecting the features and the target variable
X = file_merged[['fkstoreid', 'w.day', 'businessmonth', 'weeknum', 'trend.var', 
                 'public_holiday', 'ph_weekend', 'ramadan', 'banner.binary', 
                 'go.banner.slide', 'banner.day', 'fourteen.day.disc.price.sales.dep']]
y = file_merged['proportionUpsellrevenue']

# Handle missing values
# Option 1: Fill missing values with the mean (or median, mode, etc.)
# X = X.fillna(X.mean())

# Option 2: Drop rows with missing values
X = X.dropna()

# Add a constant to the model (intercept)
X = sm.add_constant(X)

# Build and fit the logistic regression model
model = sm.Logit(y, X)
result = model.fit()

print(result.summary())

NameError: name 'sm' is not defined

In [21]:
# Selecting the features and the target variable
X = file_merged[['fkstoreid', 'w.day', 'businessmonth', 'weeknum', 'trend.var', 
                 'public_holiday', 'ph_weekend', 'ramadan', 'banner.binary', 
                 'go.banner.slide', 'banner.day', 'fourteen.day.disc.price.sales.dep']]
y = file_merged['proportionUpsellrevenue']

# Handle missing values
# Option 1: Fill missing values with the mean (or median, mode, etc.)
X = X.fillna(X.mean())

# Option 2: Drop rows with missing values
# X = X.dropna()

# Check and handle infinite values
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Add a constant to the model (intercept)
X = sm.add_constant(X)

# Build and fit the logistic regression model
model = sm.Logit(y, X)
result = model.fit()

print(result.summary())

/var/folders/y2/30zds30n5gv0yf5nk5xk2c9c0000gn/T/ipykernel_52996/2719743508.py:9: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  X = X.fillna(X.mean())


NameError: name 'sm' is not defined

In [ ]:
################# add proportion of upselling in ####################3

In [23]:
filepath = f"/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/VK_query_results/{fileyear}/clean.sales.detail.csv"
file1 = pd.read_csv(filepath)

# Adding a column of 1 to count the number of orders each day
file1['count'] = 1

# Creating a dataframe for proportion of checks with upsell
tabulateProportion = file1.groupby(['businessdate', 'fkstoreid']).agg(upsellQnt=('quickcomboupsell', 'sum'), n=('count', 'sum')).reset_index()
tabulateProportion['pChecksWithUpsell'] = tabulateProportion['upsellQnt'] / tabulateProportion['n']

# Housekeeping the DataFrame
tabulateProportion = tabulateProportion[['businessdate', 'fkstoreid', 'pChecksWithUpsell', 'n']]

# Tabulating statistics for pChecksWithUpsell
print(f"Mean: {tabulateProportion['pChecksWithUpsell'].mean()}")
print(f"Median: {tabulateProportion['pChecksWithUpsell'].median()}")
print(f"IQR: {tabulateProportion['pChecksWithUpsell'].quantile(0.75) - tabulateProportion['pChecksWithUpsell'].quantile(0.25)}")
print(f"SD: {tabulateProportion['pChecksWithUpsell'].std()}")
print(tabulateProportion['pChecksWithUpsell'].quantile([0, 0.25, 0.5, 0.75, 1]))

# Number of Rows in the DataFrame
num_rows = len(tabulateProportion)

# Print the number of rows
print(f"Number of Rows: {num_rows}")


Mean: 0.043747282419052666
Median: 0.034746837924300306
IQR: 0.028332596226255804
SD: 0.03561370802222396
0.00    0.000000
0.25    0.022699
0.50    0.034747
0.75    0.051031
1.00    0.272860
Name: pChecksWithUpsell, dtype: float64
Number of Rows: 2602


In [ ]:
#merge the proportion of quantity and revenue together 
file = pd.merge(tabulateProportion, pUpsellrevenue)

# Displaying the first few rows of the merged DataFrame
print(file.head())

# Number of Rows in the DataFrame
num_rows = len(file)

# Print the number of rows
print(f"Number of Rows: {num_rows}")

In [ ]:
# Data Cleaning
file['businessdate'] = pd.to_datetime(file['businessdate'], format='%Y-%m-%d')

# Adding businessmonth, w.day, weeknum as factors
file['businessmonth'] = file['businessdate'].dt.month
file['w.day'] = file['businessdate'].dt.dayofweek
file['weeknum'] = ((file['businessdate'].dt.day - 1) // 7 + 1)

# Convert to categorical
file['businessmonth'] = file['businessmonth'].astype('category')
file['w.day'] = file['w.day'].astype('category')
file['weeknum'] = file['weeknum'].astype('category')

# Adding trend.var
file['trend.var'] = (file['businessdate'] - file['businessdate'].min()).dt.days + 1

In [ ]:
from datetime import datetime
import os

# Getting the public holidays data

path1 = '/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/rstudio-export-20230725/2018.public.holidays.indonesia.csv'
ph_list = pd.read_csv(path1)

# Housekeeping
ph_list['public_holiday'] = 1
ph_list['businessdate'] = pd.to_datetime(ph_list['date'], format='%m/%d/%y')
ph_list['ph_wday'] = ph_list['businessdate'].dt.day_name()
ph_list['ph_weekend'] = ph_list['public_holiday'] * ph_list['ph_wday'].isin(['Saturday', 'Sunday'])

# Selecting columns
tableToMerge = ph_list[['businessdate', 'public_holiday', 'ph_weekend']]

# Merging in PH Details
file_merged = pd.merge(file, tableToMerge, on='businessdate', how='left')

# Replacing NaN values with 0 in 'public_holiday' and 'ph_weekend'
file_merged['public_holiday'].fillna(0, inplace=True)
file_merged['ph_weekend'].fillna(0, inplace=True)

# Displaying the first few rows of the merged DataFrame
print(file_merged.head())


In [ ]:
# Adding a dummy for the Ramadan period in 2018
file_merged['ramadan'] = 0

# Creating a mask for the Ramadan period
ramadan_start = '2018-05-16'
ramadan_end = '2018-06-14'
ramadan_mask = (file_merged['businessdate'] >= ramadan_start) & (file_merged['businessdate'] <= ramadan_end)

# Applying the mask to the DataFrame
file_merged.loc[ramadan_mask, 'ramadan'] = 1

# Optional: Creating a test DataFrame for Ramadan period
ramadan_test = file_merged[['businessdate', 'ramadan']].drop_duplicates()

# Displaying the first few rows of the merged DataFrame
print(file_merged.head())

In [ ]:
import os

# Getting the promotions data
path2 = '/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/new.GoJek.Promos.csv'
banners = pd.read_csv(path2)

# Converting 'businessdate' to datetime
banners['businessdate'] = pd.to_datetime(banners['businessdate'], format='%Y-%m-%d')

# Selecting specific columns from banners
bannersToMerge = banners[['fkstoreid', 'businessdate', 'banner.binary', 'go.banner.slide', 'banner.day']]

# Displaying the first few rows of bannersToMerge
print(bannersToMerge.head())


# Converting 'businessdate' in file_merged to datetime (if not already in this format)
file_merged['businessdate'] = pd.to_datetime(file_merged['businessdate'], format='%Y-%m-%d')

# Merging file_merged with bannersToMerge
file_merged = pd.merge(file_merged, bannersToMerge, on=['fkstoreid', 'businessdate'], how='left')

# Displaying the first few rows of the merged DataFrame
print(file_merged.head())

In [ ]:
# Getting the dependence data
path3 = '/Users/jingwenshi/VK-Lab Dropbox/Maria Shi/Data/rstudio-export-20230725/2018.site.daily.platform.dependence.csv'
site_daily_dependence = pd.read_csv(path3)

# Housekeeping the Dependence Data
dependenceData = site_daily_dependence[['fkstoreid', 'businessdate', 'fourteen.day.disc.price.sales.dep']]

# Converting 'businessdate' to datetime
dependenceData['businessdate'] = pd.to_datetime(dependenceData['businessdate'])

# Displaying the first few rows of dependenceData
print(dependenceData.head())

In [ ]:
# Merging the 2 dataframes
file_merged = pd.merge(file_merged, dependenceData, on=['fkstoreid', 'businessdate'])

# Displaying the first few rows of the merged DataFrame
print(file_merged.head())

In [ ]:
# Converting to categorical data
file_merged['fkstoreid'] = file_merged['fkstoreid'].astype('category')
file_merged['sitenumber'] = file_merged['sitenumber'].astype('category')  # Assuming 'sitenumber' is in file_merged
file_merged['businessmonth'] = file_merged['businessmonth'].astype('category')  # Assuming 'businessmonth' is in file_merged
file_merged['w.day'] = file_merged['w.day'].astype('category')  # Assuming 'w.day' is in file_merged
file_merged['weeknum'] = file_merged['weeknum'].astype('category')

summary = file_merged.describe(include='all')

# Print the summary
print(summary)

# Number of Rows in the DataFrame
num_rows = len(file_merged)

# Print the number of rows
print(f"Number of Rows: {num_rows}")

In [ ]:
# Selecting the features and the target variable
X = file_merged[['fkstoreid', 'w.day', 'businessmonth', 'weeknum', 'trend.var', 
                 'public_holiday', 'ph_weekend', 'ramadan', 'banner.binary', 
                 'go.banner.slide', 'banner.day', 'fourteen.day.disc.price.sales.dep']]
y = file_merged['proportionUpsellrevenue']

# Option 1: Fill missing values with the mean (or median, mode, etc.)
# X = X.fillna(X.mean())

# Option 2: Drop rows with missing values
X.dropna(inplace=True)
y = y.loc[X.index]  # Make sure y has the same rows as X after dropping

# Add a constant to the model (intercept)
X = sm.add_constant(X)

# Weights
weights = file_merged.loc[X.index, 'n']  # Ensure weights align with X and y

# Fit the logistic regression model using GLM with a binomial family
model = sm.GLM(y, X, family=sm.families.Binomial(), weights=weights)
results = model.fit()

# Print the summary of the regression
print(results.summary())


In [ ]:
print(file_merged.columns)

#export it as excel
!pip install openpyxl
file_merged.to_excel('output_filename.xlsx', index=False, engine='openpyxl')